# GPT4All (2026 업데이트판)

[GPT4All](https://www.nomic.ai/gpt4all)은 Nomic AI가 만든 로컬 실행용 LLM 앱 및 Python 라이브러리입니다. GPU나 인터넷 연결 없이 CPU에서 동작하며, Python의 `Embed4All` 클래스로 임베딩을 만들 수 있습니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| LangChain 연동 | `langchain_community.embeddings.GPT4AllEmbeddings` | `langchain_core.embeddings.Embeddings`를 상속한 작은 래퍼로 `gpt4all.Embed4All`을 직접 사용 |
| 모델 지정 | 인자 없이 생성 (기본값: 구형 `all-MiniLM-L6-v2`, 영어 전용) | 모델명 명시 (예: `nomic-embed-text-v1.5.f16.gguf`) |
| 쿼리/문서 구분 | 없음 | Nomic 모델의 `prefix` (`search_query` / `search_document`) 적용 |
| 중복 셀 | 객체 생성 셀이 두 번 반복됨 | 제거 |

**왜 `GPT4AllEmbeddings`를 쓰지 않나요?** 이 클래스는 `langchain-community`에만 있고, 이 패키지는 2026년 5월 지원 종료되었습니다. `Embed4All`은 자체 API가 단순하므로, `Embeddings` 인터페이스로 감싸는 코드를 직접 작성하는 것이 가장 확실합니다.

**참고**: 로컬 임베딩 용도로는 현재 Ollama(05)나 Hugging Face `HuggingFaceEmbeddings`(03)가 더 널리 쓰이고 모델 선택 폭도 넓습니다. GPT4All은 데스크톱 앱(LocalDocs)과 함께 쓰는 경우에 유용합니다.

## GPT4All Python 바인딩 설치

In [ ]:
%pip install -qU gpt4all langchain-core python-dotenv numpy

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

## `Embed4All` 모델 로드

GPT4All은 CPU에 최적화된 방식으로 임의 길이의 텍스트를 임베딩합니다. 컨텍스트보다 긴 입력은 기본적으로 여러 조각으로 나눠 임베딩한 뒤 평균을 냅니다(`long_text_mode="mean"`).

- 처음 실행 시 모델 파일을 자동으로 내려받습니다(인터넷 필요, 이후에는 오프라인 동작).
- `nomic-embed-text-v1.5`는 영어 중심 모델입니다. 한국어 품질이 중요하면 03·05 노트북의 `bge-m3` 등을 고려하세요.

In [ ]:
from gpt4all import Embed4All

embed4all = Embed4All("nomic-embed-text-v1.5.f16.gguf")

## LangChain `Embeddings` 래퍼 구현

Nomic Embed 모델은 `prefix` 인자로 작업 종류를 반드시 알려야 합니다. 검색용이라면 문서에는 `search_document`, 쿼리에는 `search_query`를 사용합니다. 이 규칙을 래퍼에 넣어 두면 LangChain의 벡터 저장소나 retriever에 넘겨도 자동으로 올바른 접두어가 적용됩니다.

`dimensionality`를 지정하면 Matryoshka 방식으로 출력 차원을 줄일 수 있습니다(`nomic-embed-text-v1.5`: 64~768).

In [ ]:
from langchain_core.embeddings import Embeddings


class GPT4AllEmbedder(Embeddings):
    """gpt4all.Embed4All 을 LangChain Embeddings 인터페이스로 감싼 래퍼."""

    def __init__(self, model: Embed4All, use_nomic_prefix: bool = True, dimensionality: int | None = None):
        self.model = model
        self.use_nomic_prefix = use_nomic_prefix  # Nomic 계열 모델이 아니면 False
        self.dimensionality = dimensionality

    def _embed(self, inputs, prefix):
        return self.model.embed(
            inputs,
            prefix=prefix if self.use_nomic_prefix else None,
            dimensionality=self.dimensionality,
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts, "search_document")

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text, "search_query")


gpt4all_embd = GPT4AllEmbedder(embed4all)

In [ ]:
text = "임베딩 테스트를 하기 위한 샘플 문장입니다."  # 테스트용 문서 텍스트

## Embed the Textual Data

텍스트 임베딩의 과정은 다음과 같습니다.

1. 사전 학습된 토크나이저로 텍스트를 토큰 단위로 나누고 각 토큰을 정수 id로 바꿉니다.
2. 토큰 id를 모델에 넣어, 문맥이 반영된 토큰별 벡터를 얻습니다.
3. 토큰 벡터들을 하나로 모아(pooling, 예: 평균) 문장 전체를 나타내는 고정 길이 벡터를 만듭니다.

이 벡터는 의미 검색, 문서 분류, 군집화, 유사도 계산 등에 사용됩니다.

`embed_query`로 쿼리 임베딩을 생성하고 차원을 확인합니다.

In [ ]:
query_result = gpt4all_embd.embed_query(text)
len(query_result)  # 임베딩 차원

`embed_documents`로 여러 텍스트를 한 번에 임베딩합니다.

In [ ]:
doc_result = gpt4all_embd.embed_documents([text])
len(doc_result[0])  # 임베딩 차원

### 차원 축소 (선택)

`dimensionality`를 지정하면 더 작은 벡터를 얻습니다.

In [ ]:
gpt4all_embd_256 = GPT4AllEmbedder(embed4all, dimensionality=256)
len(gpt4all_embd_256.embed_query(text))